# 창업·벤처 녹색융합클러스터 — 신청서류 적합 검토기 (단일 노트북판)

입주 신청 구비서류(PDF/zip)를 받아 **신청요건 적합 여부를 자동 검토**하고
근거가 달린 판정 리포트(xlsx)를 생성한다.

**파이프라인:** 수집 → 분류 → 텍스트추출(텍스트레이어/OCR) → 필드추출(정규식+LLM 폴백) → 룰판정 → 리포트

**설계 원칙**
- 자동 거절 금지 — 추출 신뢰도가 낮거나 스캔 미해독이면 `부적합`이 아니라 `확인필요`.
- 최종 판정은 LLM이 아니라 **결정형 Python 규칙**(감사 가능). LLM은 필드 추출 폴백에만.
- 모든 판정에 근거(evidence)를 남긴다.

> 사용법: 위에서부터 셀을 차례로 실행 → 맨 아래 **실행 셀**에서 경로/기업명/신청일을 넣고 실행.

## 1. 패키지 설치 (최초 1회)

In [ ]:
# 파이썬 패키지
%pip install pdfplumber PyMuPDF pytesseract pdf2image openai pyyaml openpyxl pandas rapidfuzz pyzipper Pillow python-dotenv

# 한글 OCR(스캔본)용 tesseract 는 별도 설치 필요:
#   Windows : https://github.com/UB-Mannheim/tesseract/wiki  (설치 시 Korean 언어팩 체크)
#   Colab/Ubuntu :  !apt-get -y install tesseract-ocr tesseract-ocr-kor poppler-utils

## 2. 설정 (OCR / LLM 토글)

In [ ]:
import os
from types import SimpleNamespace

# (선택) 같은 폴더에 .env 가 있으면 자동 로드
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

config = SimpleNamespace(
    ENABLE_OCR=os.getenv("ENABLE_OCR", "1") == "1",
    OCR_LANG=os.getenv("OCR_LANG", "kor+eng"),
    OCR_DPI=int(os.getenv("OCR_DPI", "300")),
    OPENAI_API_KEY=os.getenv("OPENAI_API_KEY", ""),
    LLM_MODEL=os.getenv("LLM_MODEL", "gpt-4.1-mini"),
    LLM_MODEL_VISION=os.getenv("LLM_MODEL_VISION", "gpt-4.1"),
    TEXT_LAYER_MIN_CHARS=40,   # 페이지당 이 미만이면 스캔으로 간주 → OCR 경로
    ZIP_PASSWORD=os.getenv("ZIP_PASSWORD", ""),
)
config.ENABLE_LLM = bool(config.OPENAI_API_KEY)
print("OCR:", "ON" if config.ENABLE_OCR else "OFF",
      "| LLM:", ("ON(" + config.LLM_MODEL + ")") if config.ENABLE_LLM else "OFF")

### (선택) OpenAI 키를 직접 입력

In [ ]:
# LLM 필드추출 폴백을 켜려면 키를 넣고 이 셀 + 위의 '설정' 셀을 다시 실행하세요.
# 노트북을 공유하면 키가 노출되니 주의!  (가능하면 .env 사용 권장)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# config.OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]; config.ENABLE_LLM = True

## 3. 판단기준 (rules)

In [ ]:
import yaml
RULES = yaml.safe_load(r"""
# 창업·벤처 녹색융합클러스터 입주기업 신청서류 적합 판단기준
# 출처: 입주기업 모집공고(제2025-004호), 운영규정 제2·9조, 입주기업 관리지침
# 각 criterion의 check 값은 rules_engine.py 의 동일 이름 함수와 1:1로 연결된다.

meta:
  title: 신청서류 적합 판단기준
  bonus_cap: 5          # 가점 합산 최대 5점
  business_age_years: 7 # 창업 7년 이내

# 필수 공통서류(완비성 검사 대상). 서류번호는 '입주신청 구비서류' 시트 기준.
required_docs:
  - {code: "1-1", name: 사업계획서}
  - {code: "1-2", name: 사업자등록증}
  - {code: "1-3", name: 재무제표_부가세과세표준증명원}
  - {code: "1-6", name: 인감증명서}
  - {code: "1-7", name: 국세지방세납세증명서}
  - {code: "1-8", name: 4대보험가입자명부}
  - {code: "1-9", name: 개인정보수집이용동의서}
optional_docs:
  - {code: "1-4", name: 법인등기부등본, note: 법인사업자에 한함}
  - {code: "1-5", name: 주주명부, note: 법인사업자에 한함}

criteria:
  # ── 1. 대상 기업의 범위 ──
  - id: T1
    section: "1. 대상 기업의 범위"
    name: 창업 7년 이내 중소기업
    check: check_business_age
    docs: [사업자등록증, 법인등기부등본, 입주신청서]
    detail: "입주신청일 기준 창업하여 사업을 개시한 날이 7년 미경과(법인: 회사성립연월일 / 개인: 개업연월일)"
  - id: T2
    section: "1. 대상 기업의 범위"
    name: 벤처기업 자격(해당 시)
    check: check_venture
    docs: [벤처기업확인서]
    detail: "「벤처기업육성에 관한 특별조치법」 제2조의2 요건(벤처기업확인서 유효)"

  # ── 2. 제외 기업의 범위 ──
  - id: E3
    section: "2. 제외 기업의 범위"
    name: 국세·지방세 체납 등
    check: check_tax_arrears
    docs: [국세지방세납세증명서]
    detail: "납세증명서상 체납액 없음 및 유효기간 내"
  - id: E4
    section: "2. 제외 기업의 범위"
    name: 허위·부정 신청(서류 간 일치)
    check: check_consistency
    docs: [입주신청서, 사업자등록증, 법인등기부등본, 국세지방세납세증명서]
    detail: "상호·사업자등록번호·대표자가 서류 간 일치"

  # ── 완비성 ──
  - id: D0
    section: "0. 서류 완비성"
    name: 필수 공통서류 완비
    check: check_completeness
    docs: []
    detail: "필수(O) 공통서류 7종 제출 여부"

# ── 3. 가점/감점 (증빙서류 존재 → 잠정 점수, 유효성은 사람이 최종 확인) ──
bonus:
  - {id: B1, code: "2-16", name: 한국형 녹색채권 발행,        points: 3,  per_case: false, doc: 한국형녹색채권증빙}
  - {id: B2, code: "2-1",  name: 사회적기업/예비사회적기업,    points: 2,  per_case: false, doc: 사회적기업인증서}
  - {id: B3, code: "2-3~2-10", name: 신기술·환경표지·녹색·이노비즈·메인비즈·ISO14000S·KS·조달청우수제품·수출유망중소기업 등 인증, points: 2, per_case: true, doc: 인증서}
  - {id: B4, code: "2-13/2-14/2-19", name: 창업경진대회·공모전 수상/정부지원사업 선정, points: 2, per_case: true, doc: 수상_선정증빙, note: 중앙부처·지자체·대기업 주최에 한함}
  - {id: B5, code: "2-15", name: 우수환경산업체 지정,         points: 2,  per_case: true,  doc: 우수환경산업체증빙}
  - {id: B6, code: "2-18", name: 국내 특허등록·해외 특허출원,  points: 2,  per_case: true,  doc: 특허증빙}
  - {id: B7, code: "-",    name: 국가연구개발사업 제재,        points: -5, per_case: false, doc: null, note: 감점(별도 조회 필요)}

""")
print("규칙 로드:", RULES["meta"]["title"], "| 필수서류", len(RULES["required_docs"]), "종")

## 4. 파이프라인 모듈

In [ ]:
# === 수집(ingest): (중첩) zip 해제 후 PDF 목록화 ===
"""구비서류 수집: (중첩) zip 해제 후 PDF 목록화."""
import os, zipfile, tempfile, shutil

try:
    import pyzipper  # AES 암호화 zip 대응(선택)
except Exception:
    pyzipper = None


def _open_zip(path, pw):
    if pyzipper is not None:
        try:
            zf = pyzipper.AESZipFile(path)
            if pw:
                zf.setpassword(pw.encode())
            return zf
        except Exception:
            pass
    zf = zipfile.ZipFile(path)
    if pw:
        zf.setpassword(pw.encode())
    return zf


def extract_all(src_path, workdir=None, pw=""):
    """zip(중첩 포함) 또는 폴더를 받아 PDF 경로 리스트를 반환."""
    workdir = workdir or tempfile.mkdtemp(prefix="cluster_")
    pdfs = []

    def walk_zip(zpath, dest):
        os.makedirs(dest, exist_ok=True)
        with _open_zip(zpath, pw) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                try:
                    data = zf.read(info)
                except RuntimeError:
                    raise RuntimeError(f"비밀번호 오류 또는 미지정: {os.path.basename(zpath)}")
                out = os.path.join(dest, os.path.basename(info.filename))
                with open(out, "wb") as f:
                    f.write(data)
                low = out.lower()
                if low.endswith(".zip"):
                    walk_zip(out, out + "_x")
                elif low.endswith(".pdf"):
                    pdfs.append(out)

    if os.path.isdir(src_path):
        for root, _, files in os.walk(src_path):
            for fn in files:
                p = os.path.join(root, fn)
                if fn.lower().endswith(".zip"):
                    walk_zip(p, p + "_x")
                elif fn.lower().endswith(".pdf"):
                    pdfs.append(p)
    elif src_path.lower().endswith(".zip"):
        walk_zip(src_path, workdir)
    elif src_path.lower().endswith(".pdf"):
        pdfs.append(src_path)

    return sorted(set(pdfs))


In [ ]:
# === 분류(classify): 파일명 힌트 + 본문 키워드 ===
"""PDF → 서류유형(canonical) 분류. 파일명 힌트 + 본문 키워드 점수."""
import os, re

# canonical 유형: (키워드 후보)  — 본문/파일명에서 매칭
DOC_KEYWORDS = {
    "입주신청서": ["입주신청서", "입주 신청서", "녹색창업기업", "녹색벤처기업 입주신청"],
    "사업계획서": ["사업계획서", "사업 계획서"],
    "사업자등록증": ["사업자등록증", "사업자 등록증"],
    "재무제표_부가세과세표준증명원": ["부가가치세과세표준증명", "부가가치세 과세표준", "재무제표", "재무상태표", "손익계산서"],
    "법인등기부등본": ["법인등기", "등기사항전부증명서", "등기사항 전부증명서", "회사성립연월일"],
    "주주명부": ["주주명부", "주주 명부"],
    "인감증명서": ["인감증명", "법인인감", "인감 증명서"],
    "국세지방세납세증명서": ["납세증명", "국세", "지방세", "납세 증명서"],
    "4대보험가입자명부": ["4대보험", "사업장가입자명부", "가입자명부", "국민연금", "건강보험"],
    "개인정보수집이용동의서": ["개인정보", "수집", "이용 동의", "동의서"],
    # 가점 증빙
    "벤처기업확인서": ["벤처기업확인", "벤처기업 확인서"],
    "사회적기업인증서": ["사회적기업", "예비사회적기업"],
    "인증서": ["인증서", "신기술인증", "환경표지", "녹색인증", "이노비즈", "메인비즈", "ISO 14000", "KS인증", "조달청 우수제품", "수출유망"],
    "특허증빙": ["특허증", "특허등록", "특허출원", "실용신안"],
    "한국형녹색채권증빙": ["녹색채권"],
    "우수환경산업체증빙": ["우수환경산업체"],
    "수상_선정증빙": ["수상", "공모전", "경진대회", "선정", "협약서"],
}

FILENAME_HINTS = {
    "입주신청서": ["입주신청", "신청서"], "사업계획서": ["사업계획"],
    "사업자등록증": ["사업자등록"], "법인등기부등본": ["등기부", "등기"],
    "주주명부": ["주주"], "인감증명서": ["인감"],
    "국세지방세납세증명서": ["납세", "국세", "지방세"], "4대보험가입자명부": ["4대보험", "보험"],
    "개인정보수집이용동의서": ["개인정보", "동의"], "재무제표_부가세과세표준증명원": ["매출", "재무", "부가세", "과세표준"],
}


def classify(filename, text):
    name = os.path.basename(filename)
    text_head = (text or "")[:4000]
    scores = {}
    for typ, kws in DOC_KEYWORDS.items():
        s = sum(2 for kw in kws if kw in text_head)
        for kw in FILENAME_HINTS.get(typ, []):
            if kw in name:
                s += 20  # 파일명 일치는 결정적 신호(본문 내 타서류 단어 혼입을 압도)
        if s:
            scores[typ] = s
    if not scores:
        return ("미분류", 0.0)
    best = max(scores, key=scores.get)
    total = sum(scores.values())
    conf = scores[best] / total if total else 0.0
    return (best, round(conf, 2))


In [ ]:
# === 텍스트추출(extract_text): 텍스트레이어 우선, 없으면 OCR ===
"""하이브리드 추출: 텍스트 레이어가 있으면 pdfplumber, 없으면(스캔) OCR.

반환: {"text": str, "method": "text"|"ocr"|"none", "pages": int, "ocr_used": bool}
OCR는 config.ENABLE_OCR 이고 tesseract(가급적 kor 언어팩)가 있을 때만 동작.
"""
import pdfplumber


def _text_layer(pdf_path):
    parts, n = [], 0
    with pdfplumber.open(pdf_path) as pdf:
        n = len(pdf.pages)
        for pg in pdf.pages:
            t = pg.extract_text() or ""
            parts.append(t)
    return "\n".join(parts), n


def _rasterize(pdf_path, dpi):
    """PDF → PIL 이미지 리스트. pdf2image → PyMuPDF → pdfplumber 순으로 시도."""
    try:
        from pdf2image import convert_from_path
        return convert_from_path(pdf_path, dpi=dpi)
    except Exception:
        pass
    try:
        import fitz  # PyMuPDF
        imgs = []
        doc = fitz.open(pdf_path)
        zoom = dpi / 72
        for page in doc:
            pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
            from PIL import Image
            imgs.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
        return imgs
    except Exception:
        pass
    imgs = []
    with pdfplumber.open(pdf_path) as pdf:
        for pg in pdf.pages:
            imgs.append(pg.to_image(resolution=dpi).original)
    return imgs


def _ocr(pdf_path):
    import pytesseract
    text = []
    for img in _rasterize(pdf_path, config.OCR_DPI):
        text.append(pytesseract.image_to_string(img, lang=config.OCR_LANG))
    return "\n".join(text)


def extract(pdf_path):
    text, n = _text_layer(pdf_path)
    per_page = len(text.replace("\n", "")) / max(n, 1)
    if per_page >= config.TEXT_LAYER_MIN_CHARS:
        return {"text": text, "method": "text", "pages": n, "ocr_used": False}

    # 스캔으로 판단 → OCR
    if config.ENABLE_OCR:
        try:
            otext = _ocr(pdf_path)
            if otext.strip():
                return {"text": otext, "method": "ocr", "pages": n, "ocr_used": True}
        except Exception as e:
            return {"text": text, "method": "none", "pages": n, "ocr_used": False,
                    "error": f"OCR 실패: {e}"}
    return {"text": text, "method": "none", "pages": n, "ocr_used": False,
            "note": "텍스트 레이어 없음(스캔) — OCR 미적용"}


In [ ]:
# === 필드추출(extract_fields): 정규식/앵커 + LLM 폴백 ===
"""서류 텍스트에서 핵심 필드 추출. 앵커 라벨 + 정규식이 1순위, LLM은 선택 폴백."""
import re
from datetime import date

RE_BIZNO = re.compile(r"\d{3}\s*-\s*\d{2}\s*-\s*\d{5}")
RE_CORPNO = re.compile(r"\d{6}\s*-\s*\d{7}")
RE_DATE = re.compile(r"(\d{4})\s*[년.\-/]\s*(\d{1,2})\s*[월.\-/]\s*(\d{1,2})")


def _clean(s):
    return re.sub(r"\s+", " ", s).strip() if s else s


def normalize_name(s):
    """상호 비교용 정규화: 공백·괄호(영문)·법인격 표기 제거."""
    if not s:
        return ""
    s = re.sub(r"\(.*?\)", "", s)        # (ABR Co.,Ltd) 등 괄호 제거
    s = re.sub(r"[A-Za-z.,&]", "", s)     # 영문/기호 제거
    s = s.replace("주식회사", "").replace("(주)", "").replace("㈜", "")
    return re.sub(r"\s+", "", s).strip()


def parse_kdate(s):
    m = RE_DATE.search(s or "")
    if not m:
        return None
    y, mo, d = map(int, m.groups())
    try:
        return date(y, mo, d)
    except ValueError:
        return None


def _label_pos(text, label):
    """'개 업 연 월 일'처럼 글자 사이 공백이 있어도 매칭. 라벨 끝 위치 반환."""
    pat = re.compile(r"\s*".join(map(re.escape, label)))
    m = pat.search(text or "")
    return m.end() if m else -1


def _after(text, anchors, window=40):
    """anchor 라벨(공백 허용) 뒤 window 글자에서 값 후보 반환."""
    text = text or ""
    for a in anchors:
        i = _label_pos(text, a)
        if i != -1:
            seg = text[i: i + window].lstrip(" :：)]}.\t")
            return _clean(seg.split("\n")[0])
    return None


def extract_bizno(text):
    m = RE_BIZNO.search(text or "")
    return re.sub(r"\s", "", m.group()) if m else None


def extract_corpno(text):
    m = RE_CORPNO.search(text or "")
    return re.sub(r"\s", "", m.group()) if m else None


def extract_company_name(text):
    v = _after(text, ["법인명(단체명)", "상         호", "상  호", "상 호", "법인명", "상호", "기업명", "회사명", "명     칭", "명칭"])
    if v:
        v = re.split(r"(대표자|성명|등록번호|소재지|개업|사업장|주민|생년|\d)", v)[0]
        return _clean(v) or None
    return None


def extract_ceo(text):
    v = _after(text, ["대 표 자", "대표자", "대표이사", "성         명", "성  명", "성 명", "성명"])
    if v:
        v = re.split(r"(주민|생년|사업|소재|등록|상호|\d{6})", v)[0]
        return _clean(v)
    return None


def extract_open_date(text):
    # 사업자등록증: 개업연월일 ('개 업 연 월 일' 공백 허용)
    for a in ["개업연월일", "개업일"]:
        i = _label_pos(text or "", a)
        if i != -1:
            return parse_kdate(text[i:i + 60])
    return None


def extract_incorp_date(text):
    # 법인등기부: 회사성립연월일
    for a in ["회사성립연월일", "성립연월일"]:
        i = _label_pos(text or "", a)
        if i != -1:
            return parse_kdate(text[i:i + 80])
    return None


def extract_apply_date(text):
    # 입주신청서: 신청/작성일자
    for a in ["신청일", "작성일", "제출일", "년월일"]:
        i = _label_pos(text or "", a)
        if i != -1:
            d = parse_kdate(text[i:i + 60])
            if d:
                return d
    return parse_kdate(text or "")  # 최후: 문서 내 첫 날짜


def detect_tax_arrears(text):
    """납세증명서: 체납 없음/있음 판단."""
    t = (text or "").replace(" ", "")
    if not t:
        return None
    if "체납액이없" in t or "체납된사실이없" in t or "체납사실이없" in t or "체납액없음" in t:
        return "체납없음"
    if "발급목적" in t or "납세증명" in t:
        # 증명서이긴 한데 체납문구 미검출 → 확인필요
        if "체납" in t:
            return "확인필요"
        return "체납없음"
    return None


# ── 유형별 추출 ──
def extract_fields(doc_type, text):
    f = {}
    if doc_type == "사업자등록증":
        f["사업자등록번호"] = extract_bizno(text)
        f["상호"] = extract_company_name(text)
        f["대표자"] = extract_ceo(text)
        f["개업연월일"] = extract_open_date(text)
    elif doc_type == "법인등기부등본":
        f["상호"] = extract_company_name(text)
        f["법인등록번호"] = extract_corpno(text)
        f["회사성립연월일"] = extract_incorp_date(text)
    elif doc_type == "입주신청서":
        f["기업명"] = extract_company_name(text)
        f["사업자등록번호"] = extract_bizno(text)
        f["대표자"] = extract_ceo(text)
        f["신청일"] = extract_apply_date(text)
    elif doc_type == "국세지방세납세증명서":
        f["사업자등록번호"] = extract_bizno(text)
        f["상호"] = extract_company_name(text)
        f["체납상태"] = detect_tax_arrears(text)
    elif doc_type in ("인감증명서", "재무제표_부가세과세표준증명원"):
        f["사업자등록번호"] = extract_bizno(text)
        f["상호"] = extract_company_name(text)
    # 주주명부는 주주(투자조합 등)의 번호가 섞여 회사 식별필드를 추출하지 않음(존재만 신호)
    # 가점 증빙은 존재 자체를 신호로 사용
    return {k: v for k, v in f.items() if v is not None}


# ── LLM 폴백(선택) ──
def llm_extract(doc_type, text, schema_hint):
    if not config.ENABLE_LLM:
        return {}
    try:
        from openai import OpenAI
        client = OpenAI(api_key=config.OPENAI_API_KEY)
        msg = (f"다음은 '{doc_type}' 문서에서 추출한 텍스트다. "
               f"아래 필드를 JSON으로만 출력하라(없으면 null): {schema_hint}\n\n{text[:6000]}")
        r = client.chat.completions.create(
            model=config.LLM_MODEL,
            messages=[{"role": "user", "content": msg}],
            response_format={"type": "json_object"}, temperature=0)
        import json
        return json.loads(r.choices[0].message.content)
    except Exception:
        return {}


In [ ]:
# === 룰판정(rules_engine) ===
"""판정 엔진. rules.yaml의 criterion별 check 함수를 실행해 결과를 낸다.
판정값: 적합 / 부적합 / 확인필요 / 해당없음. 모든 결과에 근거(evidence)를 남긴다(감사 가능)."""
import re
from datetime import date


def _collect(record, field, sources=None):
    """서류에서 동일 필드값을 (서류, 값)으로 수집. sources 지정 시 해당 유형만."""
    out = []
    for dt, d in record["docs"].items():
        if sources and dt not in sources:
            continue
        v = d.get("fields", {}).get(field)
        if v:
            out.append((dt, v))
    return out


def _entity_type(record):
    docs = record["docs"]
    if "법인등기부등본" in docs or any(
            d.get("fields", {}).get("법인등록번호") for d in docs.values()):
        return "법인"
    return "개인"


def R(status, detail, evidence=""):
    return {"status": status, "detail": detail, "evidence": evidence}


# ── criterion checks ──
def check_completeness(record, rules):
    req = rules["required_docs"]
    present = set(record["docs"].keys())
    missing = [r["name"] for r in req if r["name"] not in present]
    if missing:
        return R("부적합", f"필수서류 누락: {', '.join(missing)}",
                 f"제출 {len(present & {r['name'] for r in req})}/{len(req)}종")
    return R("적합", "필수 공통서류 완비", f"{len(req)}/{len(req)}종 제출")


def check_business_age(record, rules):
    yrs = rules["meta"]["business_age_years"]
    et = _entity_type(record)
    apply_d = record.get("apply_date")
    docs = record["docs"]
    base, src = None, ""
    if et == "법인":
        base = docs.get("법인등기부등본", {}).get("fields", {}).get("회사성립연월일")
        src = "법인등기부 회사성립연월일"
        if not base:  # 등기부 스캔 등으로 미추출 시 개업연월일로 폴백
            base = docs.get("사업자등록증", {}).get("fields", {}).get("개업연월일")
            src = "사업자등록증 개업연월일(등기부 미추출 폴백)"
    else:
        base = docs.get("사업자등록증", {}).get("fields", {}).get("개업연월일")
        src = "사업자등록증 개업연월일"
    if not base:
        return R("확인필요", f"{src} 미추출(스캔/OCR 확인 필요)", f"구분:{et}")
    if not apply_d:
        apply_d = date.today()
    days = (apply_d - base).days
    years = days / 365.25
    ev = f"{src}={base}, 신청일={apply_d}, 경과={years:.1f}년 (구분:{et})"
    if years <= yrs:
        return R("적합", f"창업 {years:.1f}년차 (7년 이내)", ev)
    return R("부적합", f"창업 {years:.1f}년 경과 (7년 초과)", ev)


def check_venture(record, rules):
    if "벤처기업확인서" in record["docs"]:
        return R("적합", "벤처기업확인서 제출(유효기간 사람 확인)", "벤처기업 자격 경로")
    return R("해당없음", "벤처기업확인서 미제출 → 창업기업 경로로 판단", "")


def check_tax_arrears(record, rules):
    d = record["docs"].get("국세지방세납세증명서")
    if not d:
        return R("부적합", "납세증명서 미제출", "")
    state = d.get("fields", {}).get("체납상태")
    if state == "체납없음":
        return R("적합", "체납 없음(납세증명)", d.get("fields", {}).get("사업자등록번호", ""))
    if state == "체납있음":
        return R("부적합", "체납 확인됨", "")
    return R("확인필요", "체납문구 미검출(스캔/OCR 확인 필요)", f"method={d.get('method')}")


def check_consistency(record, rules):
    AUTH_BIZNO = {"사업자등록증", "입주신청서", "국세지방세납세증명서", "재무제표_부가세과세표준증명원"}
    AUTH_NAME = {"사업자등록증", "입주신청서", "국세지방세납세증명서"}
    AUTH_CEO = {"사업자등록증", "입주신청서"}

    # 1) 사업자등록번호(강한 식별자) — 불일치 시 부적합
    bizs = _collect(record, "사업자등록번호", AUTH_BIZNO)
    biz_set = {re.sub(r"\D", "", v): dt for dt, v in bizs}
    if len(biz_set) > 1:
        detail = " vs ".join(f"{k}({dt})" for k, dt in biz_set.items())
        return R("부적합", "사업자등록번호 불일치(허위·오기 의심)", detail)

    # 2) 상호·대표자 — 약한 신호(추출 노이즈). 불일치 시 확인필요
    soft = []
    names = [(dt, normalize_name(v)) for dt, v in _collect(record, "상호", AUTH_NAME)]
    names += [(dt, normalize_name(d["fields"]["기업명"])) for dt, d in record["docs"].items()
              if dt in AUTH_NAME and d.get("fields", {}).get("기업명")]
    names = [(dt, n) for dt, n in names if n]
    uniq = {n for _, n in names}
    # 포함관계면 동일로 간주(예: '에이비알' ⊂ '주식회사에이비알')
    if len(uniq) > 1 and not all(any(a in b or b in a for b in uniq if b != a) for a in uniq):
        soft.append("상호 표기 불일치: " + " vs ".join(f"{n}({dt})" for dt, n in names))

    ceos = _collect(record, "대표자", AUTH_CEO)
    ceo_set = {v.replace(" ", "") for _, v in ceos}
    if len(ceo_set) > 1:
        soft.append("대표자 불일치: " + " vs ".join(f"{v}({dt})" for dt, v in ceos))

    if soft:
        return R("확인필요", "상호/대표자 표기 차이 — 사람 확인 권장", " / ".join(soft))

    checked = []
    if bizs:
        checked.append("사업자등록번호")
    if names:
        checked.append("상호")
    if ceos:
        checked.append("대표자")
    return R("적합", "신뢰서류 간 핵심정보 일치", f"대조필드: {', '.join(checked) or '없음'}")


CHECKS = {f.__name__: f for f in
          [check_completeness, check_business_age, check_venture,
           check_tax_arrears, check_consistency]}


def evaluate_bonus(record, rules):
    rows, total = [], 0
    for b in rules.get("bonus", []):
        doc = b.get("doc")
        present = bool(doc) and doc in record["docs"]
        # 인증서는 일반 '인증서' 유형 + 사회적기업/벤처 등 세분 유형 포함
        if b["id"] == "B3":
            present = any(t in record["docs"] for t in ["인증서"])
        if b["id"] == "B2":
            present = "사회적기업인증서" in record["docs"]
        pts = b["points"] if present and b["points"] > 0 else 0
        if present and b["points"] > 0:
            total += pts
        rows.append({
            "id": b["id"], "항목": b["name"], "배점": b["points"],
            "증빙제출": "O" if present else "X",
            "잠정점수": pts if present else 0,
            "비고": ("유효성 사람 확인" if present else "") + (f" / {b['note']}" if b.get("note") else ""),
        })
    total = min(total, rules["meta"]["bonus_cap"])
    return rows, total


def evaluate(record, rules):
    results = []
    for c in rules["criteria"]:
        fn = CHECKS.get(c["check"])
        res = fn(record, rules) if fn else R("확인필요", "미구현 check", c["check"])
        results.append({"id": c["id"], "section": c["section"], "name": c["name"],
                        **res})
    bonus_rows, bonus_total = evaluate_bonus(record, rules)

    statuses = [r["status"] for r in results]
    if "부적합" in statuses:
        overall = "부적합"
    elif "확인필요" in statuses:
        overall = "확인필요"
    else:
        overall = "적합"
    return {"results": results, "bonus": bonus_rows, "bonus_total": bonus_total,
            "overall": overall}


In [ ]:
# === 리포트(report): xlsx 4시트 ===
"""판정 결과를 xlsx 총괄표로 출력."""
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

THIN = Side(style="thin")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
HEAD = PatternFill("solid", fgColor="305496")
COLORS = {"적합": "C6EFCE", "부적합": "FFC7CE", "확인필요": "FFEB9C", "해당없음": "E7E6E6"}


def _style(ws, ncol, header_row=1):
    for c in range(1, ncol + 1):
        cell = ws.cell(header_row, c)
        cell.font = Font(name="맑은 고딕", bold=True, color="FFFFFF")
        cell.fill = HEAD
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = BORDER
    for row in ws.iter_rows(min_row=header_row + 1, max_col=ncol):
        for cell in row:
            cell.font = Font(name="맑은 고딕")
            cell.alignment = Alignment(vertical="center", wrap_text=True)
            cell.border = BORDER


def build_report(company_name, record, judgment, out_path):
    wb = Workbook()

    # 1) 종합판정
    ws = wb.active; ws.title = "종합판정"
    ws.append(["기업명", company_name])
    ws.append(["종합판정", judgment["overall"]])
    ws.append(["신청일", str(record.get("apply_date") or "")])
    ws.append(["제출 PDF 수", record.get("n_pdfs")])
    ws.append(["가점(잠정 합계)", judgment["bonus_total"]])
    ws["B2"].fill = PatternFill("solid", fgColor=COLORS.get(judgment["overall"], "FFFFFF"))
    for r in range(1, 6):
        ws.cell(r, 1).font = Font(name="맑은 고딕", bold=True)
        for c in (1, 2):
            ws.cell(r, c).border = BORDER
    ws.column_dimensions["A"].width = 16; ws.column_dimensions["B"].width = 40

    # 2) 판단기준별 결과
    ws2 = wb.create_sheet("판단기준별 결과")
    ws2.append(["구분", "ID", "판단기준", "판정", "근거/세부", "비고(evidence)"])
    for r in judgment["results"]:
        ws2.append([r["section"], r["id"], r["name"], r["status"], r["detail"], r["evidence"]])
    _style(ws2, 6)
    for r in range(2, ws2.max_row + 1):
        st = ws2.cell(r, 4).value
        ws2.cell(r, 4).fill = PatternFill("solid", fgColor=COLORS.get(st, "FFFFFF"))
    for col, w in zip("ABCDEF", [20, 6, 22, 10, 45, 40]):
        ws2.column_dimensions[col].width = w

    # 3) 가점 증빙
    ws3 = wb.create_sheet("가점 증빙")
    ws3.append(["ID", "가점항목", "배점", "증빙제출", "잠정점수", "비고"])
    for b in judgment["bonus"]:
        ws3.append([b["id"], b["항목"], b["배점"], b["증빙제출"], b["잠정점수"], b["비고"]])
    ws3.append(["", "합계(최대 5점)", "", "", judgment["bonus_total"], ""])
    _style(ws3, 6)
    for col, w in zip("ABCDEF", [6, 40, 8, 10, 10, 30]):
        ws3.column_dimensions[col].width = w

    # 4) 서류 처리 로그
    ws4 = wb.create_sheet("서류 처리내역")
    ws4.append(["파일", "분류 유형", "신뢰도", "추출방식", "필드수"])
    for d in record.get("doc_log", []):
        ws4.append([d["file"], d["유형"], d["신뢰도"], d["추출방식"], d["필드수"]])
    _style(ws4, 5)
    for col, w in zip("ABCDE", [40, 24, 10, 10, 8]):
        ws4.column_dimensions[col].width = w

    wb.save(out_path)
    return out_path


## 5. 오케스트레이터

In [ ]:
import os

def process_company(src_path, apply_date=None, pw="", progress=None):
    # src_path: zip/폴더/PDF.  반환: (record, rules)
    rules = RULES
    pdfs = extract_all(src_path, pw=pw)
    docs, doc_log = {}, []
    for i, p in enumerate(pdfs):
        if progress:
            progress(i, len(pdfs), os.path.basename(p))
        ext = extract(p)                                   # 텍스트추출
        typ, conf = classify(p, ext["text"])               # 분류
        fields = extract_fields(typ, ext["text"]) if typ != "미분류" else {}
        entry = {"present": True, "file": os.path.basename(p), "method": ext["method"],
                 "confidence": conf, "fields": fields, "pages": ext.get("pages")}
        # 동일 유형 중복 시 필드가 더 많은 것을 채택
        if typ not in docs or len(fields) > len(docs[typ].get("fields", {})):
            docs[typ] = entry
        doc_log.append({"file": entry["file"], "유형": typ, "신뢰도": conf,
                        "추출방식": ext["method"], "필드수": len(fields)})
    if apply_date is None:
        apply_date = docs.get("입주신청서", {}).get("fields", {}).get("신청일")
    record = {"docs": docs, "apply_date": apply_date, "doc_log": doc_log, "n_pdfs": len(pdfs)}
    return record, rules

print("process_company 준비됨")

## 6. 실행 — 경로/기업명/신청일을 넣고 실행

In [ ]:
from datetime import date
import pandas as pd

# ▼▼▼ 여기만 바꾸세요 ▼▼▼
SRC = r"검토할_구비서류.zip"     # zip / 폴더 / PDF 경로
COMPANY = "에이비알"
APPLY_DATE = date(2026, 3, 16)    # 입주신청일
PW = "260529"                     # zip 비밀번호 (없으면 "")
# ▲▲▲ ─────────────── ▲▲▲

record, rules = process_company(SRC, apply_date=APPLY_DATE, pw=PW,
                                progress=lambda i, n, f: print(f"[{i+1}/{n}] {f}"))
judgment = evaluate(record, rules)

print("\n=== 종합판정:", judgment["overall"], "| 가점(잠정):", judgment["bonus_total"], "점 ===")
df = pd.DataFrame([{"구분": r["section"], "판단기준": r["name"], "판정": r["status"],
                    "근거/세부": r["detail"], "evidence": r["evidence"]}
                   for r in judgment["results"]])
display(df)

out = f"판정결과_{COMPANY}.xlsx"
build_report(COMPANY, record, judgment, out)
print("리포트 저장:", out)

### 처리내역 / 가점 보기 (선택)

In [ ]:
display(pd.DataFrame(record["doc_log"]))
display(pd.DataFrame(judgment["bonus"]))